# 🧠 Lab 02: CNN 文本分類 (TextCNN)

**使用 PyTorch 從零實現 TextCNN** (Kim, 2014)

---

### 🎯 學習目標
- 理解 TextCNN 架構 (Embedding → 多尺度卷積 → Pooling)
- 從零構建 PyTorch 數據集
- 訓練循環與視覺化

### 📊 數據集
**ChnSentiCorp** 子集 (5000 條)

### 🛠 技術棧
PyTorch, jieba, scikit-learn

### ⏱ 預計時間
- GPU: 強烈建議 (Colab 免費 GPU 足夠)
- 運行時間: ~5 分鐘 (含訓練)

### 🚀 GPU 加速
**菜單 → 更改執行階段類型 → 硬件加速器 → GPU**

---

## 1. 環境準備

In [ ]:
!pip install -q torch jieba pandas matplotlib scikit-learn datasets
import torch
print(f'✅ PyTorch {torch.__version__}')
print(f'✅ CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   設備: {torch.cuda.get_device_name(0)}')

## 2. 載入與預處理

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import jieba

plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'PingFang TC', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def preprocess_chinese(text):
    text = re.sub(r'[^\u4e00-\u9fffA-Za-z0-9 ]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return [w for w in jieba.lcut(text) if len(w) > 1]

def load_data():
    try:
        from datasets import load_dataset
        ds = load_dataset('lansinuote/ChnSentiCorp', split='train[:5000]')
        df = pd.DataFrame({'text': ds['text'], 'label': ds['label']})
    except Exception:
        data_dir = '/content/data' if os.path.exists('/content') else './data'
        if not os.path.exists(f'{data_dir}/sample.csv'):
            os.makedirs(data_dir, exist_ok=True)
            !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/02-text-classification-cnn/data/sample.csv -O {data_dir}/sample.csv
        df = pd.read_csv(f'{data_dir}/sample.csv')
    return df.dropna().reset_index(drop=True)

df = load_data()
print(f'📊 數據集: {len(df)} 條')
print(df['label'].value_counts())
df.head()

In [ ]:
# 分詞 + 詞表
print('🔧 分詞中...')
texts = [preprocess_chinese(t) for t in df['text']]
labels = df['label'].values

counter = Counter()
for t in texts: counter.update(t)

vocab = {'<pad>': 0, '<unk>': 1}
for word, freq in counter.most_common(30000):
    if freq >= 2:
        vocab[word] = len(vocab)
print(f'📚 詞表大小: {len(vocab)}')

## 3. 構建 DataLoader

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

MAX_LEN = 128

def encode(tokens, vocab, max_len=MAX_LEN):
    ids = [vocab.get(w, vocab['<unk>']) for w in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [vocab['<pad>']] * (max_len - len(ids))
    return ids

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = [encode(t, vocab) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.texts[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }

X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
train_loader = DataLoader(TextDataset(X_tr, y_tr, vocab), batch_size=32, shuffle=True)
test_loader = DataLoader(TextDataset(X_te, y_te, vocab), batch_size=64)
print(f'✅ Train: {len(X_tr)}, Test: {len(X_te)}')

## 4. 構建 TextCNN 模型

In [ ]:
class TextCNN(nn.Module):
    """TextCNN (Kim, 2014): Embedding → 多尺度卷積 → MaxPool → FC"""
    def __init__(self, vocab_size, embed_dim=128, n_classes=2,
                 kernel_sizes=(2, 3, 4, 5), n_filters=128, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, n_filters, kernel_size=k, padding=k//2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(kernel_sizes), n_classes)
    
    def forward(self, input_ids):
        x = self.embedding(input_ids).transpose(1, 2)
        outs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            c = F.adaptive_max_pool1d(c, 1).squeeze(-1)
            outs.append(c)
        h = torch.cat(outs, dim=1)
        h = self.dropout(h)
        return self.fc(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TextCNN(vocab_size=len(vocab), n_classes=2).to(device)
print(f'🧠 設備: {device}')
print(f'🧠 參數量: {sum(p.numel() for p in model.parameters()):,}')
model

## 5. 訓練

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
EPOCHS = 10

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        lab = batch['label'].to(device)
        optimizer.zero_grad()
        loss = criterion(model(ids), lab)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * lab.size(0)
        correct += (model(ids).argmax(1) == lab).sum().item()
        total += lab.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device)
        lab = batch['label'].to(device)
        logits = model(ids)
        loss = criterion(logits, lab)
        total_loss += loss.item() * lab.size(0)
        pred = logits.argmax(1)
        correct += (pred == lab).sum().item()
        total += lab.size(0)
        all_preds.extend(pred.cpu().tolist())
        all_labels.extend(lab.cpu().tolist())
    return total_loss/total, correct/total, all_preds, all_labels

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc, best_preds = 0, None
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc, preds, labs = evaluate(model, test_loader, criterion, device)
    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss); history['val_acc'].append(va_acc)
    print(f'Epoch {epoch:2d} | tr loss={tr_loss:.4f} acc={tr_acc:.4f} | va loss={va_loss:.4f} acc={va_acc:.4f}')
    if va_acc > best_acc:
        best_acc = va_acc; best_preds = preds

## 6. 視覺化

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

cm = confusion_matrix(labs, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['負面','正面'], yticklabels=['負面','正面'])
ax.set_title(f'Best Test Acc: {best_acc:.4f}')
plt.tight_layout(); plt.show()

## 7. 互動測試

In [ ]:
def predict(text, model, vocab, device):
    tokens = preprocess_chinese(text)
    ids = torch.tensor([encode(tokens, vocab)], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(ids)
        probs = F.softmax(logits, dim=-1)[0]
        pred = logits.argmax(1).item()
    return '😊 正面' if pred == 1 else '😞 負面', probs[pred].item()

for t in ['這部電影真的很棒', '完全浪費時間', '服務態度一流', '質量太差了']:
    s, c = predict(t, model, vocab, device)
    print(f'{t:20s} → {s} ({c:.2%})')

## 🎉 恭喜完成 Lab 02！

**你學到了**：
- ✅ PyTorch 數據集構建
- ✅ TextCNN 架構（多尺度卷積）
- ✅ 訓練循環、優化、調度
- ✅ 訓練過程視覺化

### 🚀 下一步

👉 [Lab 03: BERT 情感分析](../03-transformer-sentiment/lab.ipynb) — 預訓練模型微調

### 📚 延伸閱讀

- [Kim (2014) TextCNN 論文](https://arxiv.org/abs/1408.5882)
- [DPCNN](https://arxiv.org/abs/1709.01102)
- [TextRCNN](https://www.aaai.org/ocs/index.php/AAAI/AAAI15/paper/view/9745)